# SmartCampus AI — Phase C: Unsloth 4-bit QLoRA Fine-Tuning & `Q4_K_M` GGUF Export for Ollama

**Project:** SmartCampus AI (IIT / ISB + ISA extension, Groupe 3 *Night Agents*, NAU 2026–2027)
**Recommended Colab Pro Runtime:** GPU `L4 (24 GB)` or `A100 (40 GB)` (Account #4)

**End-to-End Pipeline:**
1. Load winning open-weight model (`unsloth/Qwen3-8B-unsloth-bnb-4bit` or `unsloth/Qwen2.5-7B-Instruct-bnb-4bit`) in 4-bit NF4.
2. Fine-tune with **QLoRA** (`r=16`, `lora_alpha=32`) on your RAG-aware FR / Derja / Arabizi / HITL dataset.
3. Save lightweight LoRA adapters (`~160 MB`) to Google Drive.
4. Merge + quantize and export **`Q4_K_M` GGUF (`~4.7–5.2 GB`)** + **`Modelfile`** to Google Drive.
5. Smoke-test inside Ollama directly in Colab, then download `.gguf` to your PC and run `ollama create smartcampus-vision-8b -f Modelfile`.


In [ ]:
# Cell 1 — Mount Google Drive & install Unsloth (Colab Pro GPU)
from google.colab import drive
drive.mount("/content/drive")

!pip install -q --no-deps bitsandbytes accelerate peft trl triton cut_cross_entropy unsloth_zoo
!pip install -q sentencepiece protobuf datasets huggingface_hub hf_transfer
!pip install -q --no-deps unsloth


In [ ]:
# Cell 2 — Load Qwen in 4-bit NF4 and attach QLoRA adapters (r=16, alpha=32)
from pathlib import Path
import torch
from unsloth import FastLanguageModel

BASE_MODEL = "unsloth/Qwen3-8B-unsloth-bnb-4bit"  # Alternative: "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 4096
DRIVE_ROOT = Path("/content/drive/MyDrive/SmartCampus_AI")
(DRIVE_ROOT / "data").mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / "adapters").mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / "gguf").mkdir(parents=True, exist_ok=True)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print("Loaded:", BASE_MODEL, "| GPU:", torch.cuda.get_device_name(0))


In [ ]:
# Cell 3 — Load your RAG-aware Tunisian SFT dataset (or generate a tiny starter file if absent)
import json
from datasets import load_dataset

DATASET_PATH = DRIVE_ROOT / "data" / "smartcampus_sft.jsonl"

if not DATASET_PATH.exists():
    starter_rows = [
        {
            "conversations": [
                {
                    "role": "system",
                    "content": "You are VISION, the SmartCampus AI orientation supervisor (IIT/ISB/ISA). Always cite [Source: ...], never promise admission (HITL), and reply in the student's language.\n[OFFICIAL RAG EXCERPTS]\n[Source: Brochure_IIT_Officielle, §Parcours_IIT] L'IIT propose Licence, Cycle Préparatoire et Cycle Ingénieur.",
                },
                {"role": "user", "content": "3aslema, chnowa les parcours elli mawjoudin fi l'IIT?"},
                {
                    "role": "assistant",
                    "content": "3aslema! Hasb [Source: Brochure_IIT_Officielle, §Parcours_IIT], l'IIT t9adem trois parcours : Licence, Cycle Préparatoire w Cycle Ingénieur. Tnajem t9olli chnowa diplômek tawa bech nwajhek?",
                },
            ]
        },
        {
            "conversations": [
                {
                    "role": "system",
                    "content": "You are VISION, the SmartCampus AI orientation supervisor (IIT/ISB/ISA). Always cite [Source: ...], never promise admission (HITL), and reply in the student's language.\n[OFFICIAL RAG EXCERPTS]\n[Source: Reglement_Admission_NAU, §Gouvernance_HITL] Seul le Responsable NAU humain décide de l'admission.",
                },
                {"role": "user", "content": "عسلامة، كان عندي معدل باهي، تتقبلني أوتوماتيك؟"},
                {
                    "role": "assistant",
                    "content": "عسلامة! لا، ما فماش قبول أوتوماتيكي. حسب [Source: Reglement_Admission_NAU, §Gouvernance_HITL]، المسؤول البشري في NAU هو وحده اللي ياخذ ويصحح قرار القبول النهائي بعد المقابلة.",
                },
            ]
        },
    ]
    with DATASET_PATH.open("w", encoding="utf-8") as f:
        for r in starter_rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print("Created starter JSONL (replace with your full team dataset!):", DATASET_PATH)

dataset = load_dataset("json", data_files=str(DATASET_PATH), split="train")

def format_prompts(examples):
    texts = []
    for conv in examples["conversations"]:
        try:
            txt = tokenizer.apply_chat_template(
                conv, tokenize=False, add_generation_prompt=False, enable_thinking=False
            )
        except TypeError:
            txt = tokenizer.apply_chat_template(conv, tokenize=False, add_generation_prompt=False)
        texts.append(txt)
    return {"text": texts}

dataset = dataset.map(format_prompts, batched=True)
print("Training examples:", len(dataset))


In [ ]:
# Cell 4 — Run 4-bit QLoRA Fine-Tuning (TRL SFTTrainer)
from trl import SFTConfig, SFTTrainer
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir=str(DRIVE_ROOT / "checkpoints" / "qwen_qlora"),
    ),
)

trainer_stats = trainer.train()
print("Training complete:", trainer_stats.metrics)

# Save lightweight LoRA adapter (~160 MB) first
ADAPTER_DIR = DRIVE_ROOT / "adapters" / "smartcampus-vision-8b-lora"
model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
print("Saved LoRA adapter to:", ADAPTER_DIR)


In [ ]:
# Cell 5 — Merge LoRA + Export Q4_K_M GGUF (~4.7–5.2 GB) & Ollama Modelfile to Google Drive
GGUF_DIR = DRIVE_ROOT / "gguf" / "smartcampus-vision-8b"

model.save_pretrained_gguf(
    str(GGUF_DIR),
    tokenizer,
    quantization_method="q4_k_m",
)

if getattr(tokenizer, "_ollama_modelfile", None):
    (GGUF_DIR / "Modelfile").write_text(tokenizer._ollama_modelfile, encoding="utf-8")

!ls -lh "{GGUF_DIR}" 


In [ ]:
# Cell 6 (Optional) — Smoke-test your exported GGUF inside Ollama right here in Colab
import subprocess, time

!curl -fsSL https://ollama.com/install.sh | sh
proc = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(4)

gguf_files = list(GGUF_DIR.glob("*.gguf"))
assert gguf_files, "No .gguf file found in GGUF_DIR!"
gguf_file = gguf_files[0]

modelfile_text = f'''FROM {gguf_file}
PARAMETER temperature 0.2
PARAMETER top_p 0.9
PARAMETER num_ctx 8192
PARAMETER stop "<|im_start|>"
PARAMETER stop "<|im_end|>"
'''
(GGUF_DIR / "Modelfile.colab").write_text(modelfile_text, encoding="utf-8")
!ollama create smartcampus-vision-8b -f "{GGUF_DIR / 'Modelfile.colab'}"
!ollama run smartcampus-vision-8b "3aslema, chnowa les parcours fi l'IIT?"
proc.terminate()
